# Pooling

**Capítulo 4 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_convolutional-neural-networks/pooling.ipynb` · [Lección original](https://d2l.ai/chapter_convolutional-neural-networks/pooling.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Pooling
<a id="sec_pooling"></a>

En muchos casos, nuestra última tarea plantea una pregunta global sobre la imagen, por ejemplo, *¿contiene un gato?* Por lo tanto, las unidades de nuestra capa final deben ser sensibles a toda la entrada. Agregando gradualmente la información, produciendo mapas más gruesos y más gruesos, logramos este objetivo de finalmente aprender una representación global, manteniendo al mismo tiempo todas las ventajas de las capas convolucionales en las capas intermedias de procesamiento. Cuanto más profundo vamos en la red, mayor es el campo receptivo (relativo a la entrada) al que cada nodo oculto es sensible. La reducción de la resolución espacial acelera este proceso, ya que los núcleos de la convolución cubren un área efectiva más grande.

Por otra parte, al detectar características de nivel inferior, como los bordes (como se discute en [Referencia sec_conv_layer](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html#sec-conv-layer)), a menudo queremos que nuestras representaciones sean algo invariantes a la traducción. Por ejemplo, si tomamos la imagen `X` con una delineación nítida entre blanco y negro y desplazamos toda la imagen por un píxel a la derecha, es decir, `Z[i, j] = X[i, j + 1]`, entonces la salida de la nueva imagen `Z` podría ser muy diferente. El borde se habrá desplazado por un píxel. En realidad, los objetos casi nunca ocurren exactamente en el mismo lugar. De hecho, incluso con un trípode y un objeto estacionario, la vibración de la cámara debido al movimiento del obturador puede cambiar todo por un píxel o así (cámaras de gama alta están cargadas con características especiales para abordar este problema).

Esta sección introduce *capas de agrupación*, que sirven al doble propósito de mitigar la sensibilidad de las capas convolucionales a la ubicación y de las representaciones espacialmente descendentes.


In [ ]:
import torch
from torch import nn
from laboratorio import d2l

## Max pooling y average pooling
Al igual que las capas convolucionales, los operadores *pooling* consisten en una ventana de forma fija que se desliza sobre todas las regiones de la entrada de acuerdo con su paso, computando una única salida para cada ubicación atravesada por la ventana de forma fija (a veces conocida como la ventana *pooling*). Sin embargo, a diferencia del cálculo de correlación cruzada de las entradas y núcleos en la capa convolucional, la capa de pooling no contiene parámetros (no hay *kernel*). En cambio, los operadores de pooling son deterministas, calculando normalmente el valor máximo o medio de los elementos en la ventana de pooling. Estas operaciones se llaman *pooling máximo* (*max-pooling* para corto) y *pooling medio*, respectivamente.

*El pooling promedio* es esencialmente tan antiguo como CNNs. La idea es similar a la toma de muestras de una imagen. En lugar de tomar el valor de cada segundo (o tercer) píxel para la imagen de resolución inferior, podemos promedio sobre píxeles adyacentes para obtener una imagen con una mejor relación señal-ruido ya que estamos combinando la información de varios píxeles adyacentes. *Max-pooling* fue introducido en
[Riesenhuber.Poggio.1999](https://d2l.ai/chapter_references/zreferences.html) en el contexto de la neurociencia cognitiva para describir 
cómo la agregación de información puede ser agregada jerárquicamente a efectos de reconocimiento de objetos; ya había una versión anterior en el reconocimiento de voz [Yamaguchi.Sakamoto.Akabane.ea.1990](https://d2l.ai/chapter_references/zreferences.html). En casi todos los casos, max-pooling, como también se menciona, es preferible a la pooling media.

En ambos casos, al igual que con el operador de correlación cruzada, podemos pensar en la ventana de pooling como a partir de la parte superior izquierda del tensor de entrada y deslizarse a través de él de izquierda a derecha y de arriba a abajo. En cada ubicación que la ventana de pooling golpea, se calcula el valor máximo o medio del subtensor de entrada en la ventana, dependiendo de si se utiliza max o media pooling.

![Max-pooling con ventana $2\times 2$. La región resaltada produce la primera salida: $\max(0,1,3,4)=4$.](../recursos/originales/pooling.svg)
<a id="fig_pooling"></a>

El tensor de salida en [Referencia fig_pooling](https://d2l.ai/chapter_convolutional-neural-networks/pooling.html#fig-pooling) tiene una altura de 2 y una anchura de 2. Los cuatro elementos se derivan del valor máximo en cada ventana de pooling:

$$
\max(0, 1, 3, 4)=4,\\
\max(1, 2, 4, 5)=5,\\
\max(3, 4, 6, 7)=7,\\
\max(4, 5, 7, 8)=8.\\
$$

De forma más general, podemos definir una capa de agrupación $p \times q$ agregándose sobre una región de dicho tamaño. Volviendo al problema de la detección de bordes, utilizamos la salida de la capa convolucional como entrada para la agrupación máxima $2\times 2$. Denota por `X` la entrada de la entrada de la capa convolucional y `Y` la salida de la capa de agrupación. Independientemente de si los valores de `X[i, j]`, `X[i, j + 1]`, `X[i+1, j]` y `X[i+1, j + 1]` son diferentes, la capa de agrupación siempre sale `Y[i, j] = 1`. Es decir, usando la capa de agrupación máxima $2\times 2$, todavía podemos detectar si el patrón reconocido por la capa convolucional no se mueve más de un elemento en altura o anchura.

En el código siguiente, **aplicamos la propagación hacia delante de la capa de pooling** en la función `pool2d`. Esta función es similar a la función `corr2d` en [Referencia sec_conv_layer](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html#sec-conv-layer). Sin embargo, no se necesita núcleo, computando la salida como el máximo o el promedio de cada región en la entrada.


In [ ]:
def pool2d(X, pool_size, mode='max'):
    p_h, p_w = pool_size
    Y = torch.zeros((X.shape[0] - p_h + 1, X.shape[1] - p_w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i, j] = X[i: i + p_h, j: j + p_w].max()
            elif mode == 'avg':
                Y[i, j] = X[i: i + p_h, j: j + p_w].mean()
    return Y

Podemos construir el tensor de entrada `X` en [Referencia fig_pooling](https://d2l.ai/chapter_convolutional-neural-networks/pooling.html#fig-pooling) para **validar la salida de la capa bidimensional de max-pooling**.


In [ ]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
pool2d(X, (2, 2))

Además, podemos experimentar con **la capa media de pooling**.


In [ ]:
pool2d(X, (2, 2), 'avg')

## Padding y Stride

Al igual que con las capas convolucionales, las capas de unión cambian la forma de salida. Y como antes, podemos ajustar la operación para lograr una forma de salida deseada acolchando la entrada y ajustando el paso. Podemos demostrar el uso de padding y pasos en las capas de agrupación a través de la capa de unión máxima bidimensional incorporada desde el biblioteca de aprendizaje profundo. Primero construimos un tensor de entrada `X` cuya forma tiene cuatro dimensiones, donde el número de ejemplos (tamaño de lote) y el número de canales son ambos 1.


In [ ]:
X = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))
X

Dado que la combinación de información agregada de un área, **bibliotecas de aprendizaje profundos por defecto a la combinación de tamaños y pasos de ventana de pooling.** Por ejemplo, si usamos una ventana de pooling de forma `(3, 3)` obtenemos una forma zancada de `(3, 3)` por defecto.


### Nota docente de Hespérides

Comprueba primero las formas y el supuesto arquitectónico: localidad, compartición de pesos o conexión residual. El explorador permite seguir ventana, multiplicaciones y suma. En PyTorch, Conv2d implementa correlación cruzada; en aprendizaje profundo se suele llamar convolución a esta operación. El autoencoder 91 amplía el patrón MLP con reconstrucción; VAE se trata como contraste conceptual, al no existir un original válido en las fuentes locales.

Vínculo con los apuntes: sesión 4, «Pooling».


In [ ]:
pool2d = nn.MaxPool2d(3)
# La agrupación no tiene parámetros de modelo, por lo que no necesita inicialización
pool2d(X)

No hace falta decir que **el paso y el relleno se pueden especificar manualmente** para anular los valores predeterminados de framework si es necesario.


In [ ]:
pool2d = nn.MaxPool2d(3, padding=1, stride=2)
pool2d(X)

Por supuesto, podemos especificar una ventana de agrupación rectangular arbitraria con altura y anchura arbitrarias respectivamente, como muestra el ejemplo siguiente.


In [ ]:
pool2d = nn.MaxPool2d((2, 3), stride=(2, 3), padding=(0, 1))
pool2d(X)

## Múltiples canales
Al procesar datos de entrada multicanal, **la capa de pooling agrupa cada canal de entrada por separado**, en lugar de sumar las entradas sobre los canales como en una capa convolucional. Esto significa que el número de canales de salida para la capa de pooling es el mismo que el número de canales de entrada. A continuación, concatenaremos los tensores `X` y `X + 1` en la dimensión del canal para construir una entrada con dos canales.


In [ ]:
X = torch.cat((X, X + 1), 1)
X

Como podemos ver, el número de canales de salida sigue siendo dos después de la pooling.


In [ ]:
pool2d = nn.MaxPool2d(3, padding=1, stride=2)
pool2d(X)

## Resumen
El agrupamiento es una operación extremadamente simple. Hace exactamente lo que su nombre indica, agrega resultados sobre una ventana de valores. Toda la semántica de convolución, como los pasos y el relleno se aplican de la misma manera que lo hicieron anteriormente. Tenga en cuenta que el agrupamiento es indiferente a los canales, es decir, deja el número de canales sin cambios y se aplica a cada canal por separado. Por último, de las dos opciones populares del agrupamiento, el max-pooling es preferible al pooling promedio, ya que confiere cierto grado de invarianza a la salida. Una opción popular es elegir un tamaño de ventana de pooling de $2 \times 2$ al cuarto de la resolución espacial de la salida.

Tenga en cuenta que hay muchas más formas de reducir la resolución más allá de la pooling. Por ejemplo, en la pooling estocástica [Zeiler.Fergus.2013](https://d2l.ai/chapter_references/zreferences.html) y la agregación fraccionaria max-pooling [Graham.2014](https://d2l.ai/chapter_references/zreferences.html) se combina con la aleatorización. Esto puede mejorar ligeramente la precisión en algunos casos. Por último, como veremos más adelante con el mecanismo de atención, hay formas más refinadas de agregar sobre salidas, por ejemplo, mediante el uso de la alineación entre una consulta y vectores de representación.

## Ejercicios
1. Implementar la pooling media a través de una convolución.
1. Demostrar que el max-pooling no se puede implementar solo a través de una convolución.
1. La pooling máxima se puede realizar mediante operaciones ReLU, es decir, $\textrm{ReLU}(x) = \max(0, x)$.
    1. Expreso $\max (a, b)$ usando sólo operaciones ReLU.
    1. Utilice esto para implementar max-pooling mediante convoluciones y capas ReLU.
    1. ¿Cuántos canales y capas necesita para una convolución $2 \times 2$? ¿Cuántos para una convolución $3 \times 3$?
1. ¿Cuál es el costo computacional de la capa de pooling? Supongamos que la entrada a la capa de pooling es de tamaño $c\times h\times w$, la ventana de pooling tiene una forma de $p_\textrm{h}\times p_\textrm{w}$ con un relleno de $(p_\textrm{h}, p_\textrm{w})$ y un paso de $(s_\textrm{h}, s_\textrm{w})$.
1. ¿Por qué esperas que el max-pooling y el pooling promedio funcionen de manera diferente?
1. ¿Necesitamos una capa de agrupación mínima separada? ¿Puede reemplazarla con otra operación?
1. Podríamos usar la operación softmax para unirnos. ¿Por qué no podría ser tan popular?


[Debate del original](https://discuss.d2l.ai/t/72)
